<a href="https://colab.research.google.com/github/choruew/week7-8/blob/main/week7_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#套件安裝
!pip install -q \
    pypdf \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface-hub \
    tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.0 MB/s eta 0:00:00


In [2]:
#資料夾建立
from pathlib import Path

BASE_DIR = Path("/content/scholarship_rag")
DATA_DIR = BASE_DIR / "data"
VECTOR_DB_DIR = BASE_DIR / "vector_DB"

DATA_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

print("專案資料夾：", BASE_DIR)
print("PDF 資料夾：", DATA_DIR)
print("向量資料夾：", VECTOR_DB_DIR)

專案資料夾： /content/scholarship_rag
PDF 資料夾： /content/scholarship_rag/data
向量資料夾： /content/scholarship_rag/vector_DB


In [3]:
#上傳獎學金PDF
from google.colab import files
import shutil

print("請選擇獎學金 PDF，可一次選擇多個檔案。")

uploaded = files.upload()

for filename in uploaded.keys():
    source_path = Path("/content") / filename
    target_path = DATA_DIR / filename

    shutil.move(str(source_path), str(target_path))

print("\n上傳完成。")

請選擇獎學金 PDF，可一次選擇多個檔案。


Saving 大立光電股份有限公司獎學金.pdf to 大立光電股份有限公司獎學金.pdf
Saving 似鳥(NITORI)國際獎學金.pdf to 似鳥(NITORI)國際獎學金.pdf
Saving 助學功德金.pdf to 助學功德金.pdf
Saving 林劉金珠女士勤學獎學金.pdf to 林劉金珠女士勤學獎學金.pdf
Saving 武志先生獎助學金.pdf to 武志先生獎助學金.pdf
Saving 南加州校友會獎學金.pdf to 南加州校友會獎學金.pdf
Saving 建程人文社會獎學金.pdf to 建程人文社會獎學金.pdf
Saving 建程科學科技工程數學獎學金.pdf to 建程科學科技工程數學獎學金.pdf
Saving 書卷獎.pdf to 書卷獎.pdf
Saving 校友總會獎助學金.pdf to 校友總會獎助學金.pdf
Saving 張慧高女士紀念獎助學金.pdf to 張慧高女士紀念獎助學金.pdf
Saving 陳守先生紀念獎助學金.pdf to 陳守先生紀念獎助學金.pdf
Saving 勤學獎助學金.pdf to 勤學獎助學金.pdf
Saving 達達國際企業股份有限公司（Lagoon）弱勢獎學金.pdf to 達達國際企業股份有限公司（Lagoon）弱勢獎學金.pdf
Saving 管理學院創業學分學程獎助學金.pdf to 管理學院創業學分學程獎助學金.pdf
Saving 學生學術論文獎勵.pdf to 學生學術論文獎勵.pdf
Saving 興翼獎學金.pdf to 興翼獎學金.pdf
Saving 麗裕慈善基金會獎學金.pdf to 麗裕慈善基金會獎學金.pdf

上傳完成。


In [4]:
#檢查PDF
pdf_files = sorted(
    [
        file_path
        for file_path in DATA_DIR.iterdir()
        if file_path.is_file()
        and file_path.suffix.lower() == ".pdf"
    ],
    key=lambda path: path.name.lower()
)

print("=" * 70)
print("PDF 檔案檢查")
print("=" * 70)

print(f"PDF 數量：{len(pdf_files)}")

for index, pdf_path in enumerate(pdf_files, start=1):
    file_size_mb = pdf_path.stat().st_size / 1024**2

    print(
        f"{index:02d}. {pdf_path.name} "
        f"({file_size_mb:.2f} MB)"
    )

if not pdf_files:
    raise FileNotFoundError(
        "data 資料夾中沒有找到 PDF，請重新執行上傳 Cell。"
    )

PDF 檔案檢查
PDF 數量：18
01. 似鳥(NITORI)國際獎學金.pdf (0.29 MB)
02. 助學功德金.pdf (0.32 MB)
03. 勤學獎助學金.pdf (0.23 MB)
04. 南加州校友會獎學金.pdf (0.23 MB)
05. 大立光電股份有限公司獎學金.pdf (0.19 MB)
06. 學生學術論文獎勵.pdf (0.25 MB)
07. 建程人文社會獎學金.pdf (0.22 MB)
08. 建程科學科技工程數學獎學金.pdf (0.31 MB)
09. 張慧高女士紀念獎助學金.pdf (0.17 MB)
10. 書卷獎.pdf (0.13 MB)
11. 林劉金珠女士勤學獎學金.pdf (0.46 MB)
12. 校友總會獎助學金.pdf (0.22 MB)
13. 武志先生獎助學金.pdf (0.20 MB)
14. 管理學院創業學分學程獎助學金.pdf (0.20 MB)
15. 興翼獎學金.pdf (0.27 MB)
16. 達達國際企業股份有限公司（Lagoon）弱勢獎學金.pdf (0.26 MB)
17. 陳守先生紀念獎助學金.pdf (0.17 MB)
18. 麗裕慈善基金會獎學金.pdf (0.16 MB)


In [5]:
#測試PDF檔案是否能被讀取
from pypdf import PdfReader

print("=" * 70)
print("PDF 讀取測試")
print("=" * 70)

total_pages = 0
successful_files = 0
failed_files = []

for pdf_path in pdf_files:
    try:
        reader = PdfReader(str(pdf_path))
        page_count = len(reader.pages)

        total_pages += page_count
        successful_files += 1

        first_page_text = ""

        if page_count > 0:
            first_page_text = (
                reader.pages[0].extract_text() or ""
            ).strip()

        print(f"\n檔案：{pdf_path.name}")
        print(f"頁數：{page_count}")
        print(f"第一頁文字長度：{len(first_page_text)}")

        if first_page_text:
            preview = first_page_text[:150].replace("\n", " ")
            print(f"文字預覽：{preview}...")
        else:
            print("警告：第一頁沒有擷取到文字。")

    except Exception as error:
        failed_files.append(
            {
                "file": pdf_path.name,
                "error": str(error),
            }
        )

        print(f"\n讀取失敗：{pdf_path.name}")
        print(type(error).__name__, error)

print("\n" + "=" * 70)
print("PDF 讀取結果")
print("=" * 70)

print(f"成功檔案數：{successful_files}")
print(f"總頁數：{total_pages}")
print(f"失敗檔案數：{len(failed_files)}")

PDF 讀取測試

檔案：似鳥(NITORI)國際獎學金.pdf
頁數：4
第一頁文字長度：2312
文字預覽：國立中興大學似鳥(NITORI)國際獎學金辦法  National Chung Hsing University  Nitori International Scholarship Regulations  中華民國 108 年 03 月 07 日訂定  Established on March 7...

檔案：助學功德金.pdf
頁數：3
第一頁文字長度：2340
文字預覽：1   國立中興大學助學功德金設置辦法  National Chung Hsing University Financial Aid Regulation  96.6.27第329次行政會議訂定  Passed by Administrative meeting dated June 27, 200...

檔案：勤學獎助學金.pdf
頁數：4
第一頁文字長度：2167
文字預覽：國立中興大學勤學獎助學金授與辦法  National Chung Hsing University  Underprivileged Diligent Students Scholarship Regulations  94.6.22 第 313 次行政會議訂定  Established on Ju...

檔案：南加州校友會獎學金.pdf
頁數：3
第一頁文字長度：2116
文字預覽：國立中興大學南加州校友會獎學金辦法  National Chung Hsing University   Southern California Alumni Association Scholarship Regulations    109.08.25 南加州校友會理事會修正通過 (修訂第2,3...

檔案：大立光電股份有限公司獎學金.pdf
頁數：2
第一頁文字長度：1726
文字預覽：國立中興大學代辦大立光電股份有限公司獎學金辦法  National Chung Hsing University    LARGAN PRECISION CO., LTD. Scholarship Regulations    中華民國 96 年 05 月 28 日訂定   Established ...

檔案：學生學術論文獎勵.

In [6]:
#設定切割參數
from pathlib import Path

# ========= Chunk 設定 =========
CHUNK_SIZE = 600
CHUNK_OVERLAP = 100

print("=" * 70)
print("Chunk 參數")
print("=" * 70)

print(f"Chunk Size：{CHUNK_SIZE}")
print(f"Chunk Overlap：{CHUNK_OVERLAP}")

Chunk 參數
Chunk Size：600
Chunk Overlap：100


In [7]:
#開始切割
import json
from pypdf import PdfReader

chunks = []

chunk_id = 0

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):

    reader = PdfReader(str(pdf_path))

    print(f"處理：{pdf_path.name}")

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text is None:
            continue

        text = text.replace("\n", " ")
        text = " ".join(text.split())

        if len(text) == 0:
            continue

        start = 0
        chunk_number = 1

        while start < len(text):

            end = start + CHUNK_SIZE

            chunk_text = text[start:end]

            chunks.append(
                {
                    "id": chunk_id,
                    "source": pdf_path.name,
                    "page": page_number,
                    "chunk_number": chunk_number,
                    "text": chunk_text
                }
            )

            chunk_id += 1
            chunk_number += 1

            start += CHUNK_SIZE - CHUNK_OVERLAP

print("\n完成！")
print(f"Chunk 數量：{len(chunks)}")

處理：似鳥(NITORI)國際獎學金.pdf
處理：助學功德金.pdf
處理：勤學獎助學金.pdf
處理：南加州校友會獎學金.pdf
處理：大立光電股份有限公司獎學金.pdf
處理：學生學術論文獎勵.pdf
處理：建程人文社會獎學金.pdf
處理：建程科學科技工程數學獎學金.pdf
處理：張慧高女士紀念獎助學金.pdf
處理：書卷獎.pdf
處理：林劉金珠女士勤學獎學金.pdf
處理：校友總會獎助學金.pdf
處理：武志先生獎助學金.pdf
處理：管理學院創業學分學程獎助學金.pdf
處理：興翼獎學金.pdf
處理：達達國際企業股份有限公司（Lagoon）弱勢獎學金.pdf
處理：陳守先生紀念獎助學金.pdf
處理：麗裕慈善基金會獎學金.pdf

完成！
Chunk 數量：194


In [8]:
#檢查一下chunk內的內容
print("=" * 70)
print("第一個 Chunk")
print("=" * 70)

print(json.dumps(
    chunks[0],
    indent=4,
    ensure_ascii=False
))

print("\n")

print("=" * 70)
print("最後一個 Chunk")
print("=" * 70)

print(json.dumps(
    chunks[-1],
    indent=4,
    ensure_ascii=False
))

第一個 Chunk
{
    "id": 0,
    "source": "似鳥(NITORI)國際獎學金.pdf",
    "page": 1,
    "chunk_number": 1,
    "text": "國立中興大學似鳥(NITORI)國際獎學金辦法 National Chung Hsing University Nitori International Scholarship Regulations 中華民國 108 年 03 月 07 日訂定 Established on March 7, 2019 中華民國 108 年 03 月 19 日修正 Revised on March 19, 2019 中華民國 109 年 01 月 16 日修正 Revised on January 16, 2020 中華民國 109 年 11 月 10 日修正 Revised on November 10, 2020 中華民國 110 年 11 月 23 日修正 Revised on November 23, 2021 中華民國 111 年 12 月 12 日修正 Revised on December 12, 2022 中華民國 113 年 01 月 08 日修正 Revised on January 08, 2024 中華民國 113 年 12 月10 日修正 Revised on December 10, 2024 中華民國 115年02 月02 日修正 Revised on February 02, 2026 第一條 為實現各國年輕學子的求學之夢，從而達到與各國的友好親善以及國際化人才培育"
}


最後一個 Chunk
{
    "id": 193,
    "source": "麗裕慈善基金會獎學金.pdf",
    "page": 2,
    "chunk_number": 3,
    "text": "nscript from their previous school). 第七條 學生獲獎紀錄永久保存，申請資料則保存 1 年。 Article 7 The records of awarded students are kept permanently, while application materials are kept for 

In [9]:
#將chunk存成json檔案
chunk_file = VECTOR_DB_DIR / "chunks.json"

with open(
    chunk_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "total_chunks": len(chunks),
            "chunks": chunks
        },
        f,
        ensure_ascii=False,
        indent=4
    )

print("chunks.json 已儲存")
print(chunk_file)

chunks.json 已儲存
/content/scholarship_rag/vector_DB/chunks.json


In [10]:
#將chunk和Embedding Model載入
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer

BASE_DIR = Path("/content/scholarship_rag")
VECTOR_DB_DIR = BASE_DIR / "vector_DB"
CHUNK_FILE = VECTOR_DB_DIR / "chunks.json"

with open(CHUNK_FILE, "r", encoding="utf-8") as f:
    chunk_data = json.load(f)

chunks = chunk_data["chunks"]

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/"
    #能處理中文語意相似度的Embedding Model，能將中文文字轉成數字向量
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("=" * 70)
print("載入 Embedding Model")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print("模型名稱：", EMBEDDING_MODEL_NAME)
print("Chunk 數量：", len(chunks))
print("模型載入成功。")

載入 Embedding Model


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

模型名稱： sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Chunk 數量： 194
模型載入成功。


In [11]:
#這邊不只有chunk內容，也把來源的檔案名稱放進去
from pathlib import Path

embedding_texts = []

for chunk in chunks:
    scholarship_name = Path(chunk["source"]).stem

    embedding_text = (
        f"獎學金文件名稱：{scholarship_name}\n"
        f"文件內容：{chunk['text']}"
    )

    embedding_texts.append(embedding_text)

print("=" * 70)
print("Embedding 文字準備完成")
print("=" * 70)

print("文字數量：", len(embedding_texts))
print("\n第一筆 Embedding 文字：")
print(embedding_texts[0][:500])

Embedding 文字準備完成
文字數量： 194

第一筆 Embedding 文字：
獎學金文件名稱：似鳥(NITORI)國際獎學金
文件內容：國立中興大學似鳥(NITORI)國際獎學金辦法 National Chung Hsing University Nitori International Scholarship Regulations 中華民國 108 年 03 月 07 日訂定 Established on March 7, 2019 中華民國 108 年 03 月 19 日修正 Revised on March 19, 2019 中華民國 109 年 01 月 16 日修正 Revised on January 16, 2020 中華民國 109 年 11 月 10 日修正 Revised on November 10, 2020 中華民國 110 年 11 月 23 日修正 Revised on November 23, 2021 中華民國 111 年 12 月 12 日修正 Revised on December 12, 2022 中華民國 113 年 01 月 08 日修正 Revised on January 08, 2024 中華民國 113 年 


In [12]:
#開始建立Embedding
import numpy as np

print("=" * 70)
print("開始建立 Embedding")
print("=" * 70)

embeddings = embedding_model.encode(
    embedding_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

embeddings = embeddings.astype("float32")

print("\nEmbedding 建立完成。")
print("向量數量：", embeddings.shape[0])
print("向量維度：", embeddings.shape[1])
print("資料型態：", embeddings.dtype)

開始建立 Embedding


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Embedding 建立完成。
向量數量： 194
向量維度： 384
資料型態： float32


In [13]:
#由於後面要計算Cosine Similarity，所以要先確認向量是否已正規化
vector_norms = np.linalg.norm(embeddings, axis=1)

print("=" * 70)
print("向量正規化檢查")
print("=" * 70)

print("最小向量長度：", vector_norms.min())
print("最大向量長度：", vector_norms.max())
print("平均向量長度：", vector_norms.mean())

if np.allclose(vector_norms, 1.0, atol=1e-4):
    print("檢查通過：向量皆已正規化。")
else:
    print("警告：部分向量可能沒有正規化。")

向量正規化檢查
最小向量長度： 0.9999999
最大向量長度： 1.0000001
平均向量長度： 1.0
檢查通過：向量皆已正規化。


In [14]:
#儲存 Embedding
import json
import numpy as np

EMBEDDINGS_FILE = VECTOR_DB_DIR / "embeddings.npy"
EMBEDDING_INFO_FILE = VECTOR_DB_DIR / "embedding_info.json"

np.save(
    EMBEDDINGS_FILE,
    embeddings
)

embedding_info = {
    "model_name": EMBEDDING_MODEL_NAME,
    "total_vectors": int(embeddings.shape[0]),
    "vector_dimension": int(embeddings.shape[1]),
    "dtype": str(embeddings.dtype),
    "normalized": True,
    "embedding_text_format": (
        "獎學金文件名稱：{source_stem}\\n"
        "文件內容：{chunk_text}"
    )
}

with open(
    EMBEDDING_INFO_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        embedding_info,
        f,
        ensure_ascii=False,
        indent=4
    )

print("=" * 70)
print("Embedding 儲存完成")
print("=" * 70)

print("向量檔案：", EMBEDDINGS_FILE)
print("設定檔案：", EMBEDDING_INFO_FILE)

Embedding 儲存完成
向量檔案： /content/scholarship_rag/vector_DB/embeddings.npy
設定檔案： /content/scholarship_rag/vector_DB/embedding_info.json


In [15]:
#確認上一步是否有成功儲存
loaded_embeddings = np.load(EMBEDDINGS_FILE)

with open(
    EMBEDDING_INFO_FILE,
    "r",
    encoding="utf-8"
) as f:
    loaded_embedding_info = json.load(f)

print("=" * 70)
print("Embedding 檔案確認")
print("=" * 70)

print("重新載入形狀：", loaded_embeddings.shape)
print("模型名稱：", loaded_embedding_info["model_name"])
print("向量數量：", loaded_embedding_info["total_vectors"])
print("向量維度：", loaded_embedding_info["vector_dimension"])

if loaded_embeddings.shape[0] != len(chunks):
    raise ValueError(
        "Embedding 數量與 Chunk 數量不一致。"
    )

print("檢查通過：Embedding 數量與 Chunk 數量一致。")

Embedding 檔案確認
重新載入形狀： (194, 384)
模型名稱： sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
向量數量： 194
向量維度： 384
檢查通過：Embedding 數量與 Chunk 數量一致。


In [16]:
#建立向量資料庫(FAISS)
import json
import numpy as np
import faiss
from pathlib import Path

# 重新設定路徑，避免 Colab 重跑時發生 NameError
BASE_DIR = Path("/content/scholarship_rag")
VECTOR_DB_DIR = BASE_DIR / "vector_DB"

EMBEDDINGS_FILE = VECTOR_DB_DIR / "embeddings.npy"
FAISS_INDEX_FILE = VECTOR_DB_DIR / "faiss.index"
FAISS_INFO_FILE = VECTOR_DB_DIR / "faiss_info.json"

# 重新載入 Embedding
embeddings = np.load(EMBEDDINGS_FILE).astype("float32")

vector_count, vector_dimension = embeddings.shape

print("=" * 70)
print("建立 FAISS Index")
print("=" * 70)
print("向量數量：", vector_count)
print("向量維度：", vector_dimension)

# 建立 Inner Product Index
faiss_index = faiss.IndexFlatIP(vector_dimension)

# 將向量加入 FAISS
faiss_index.add(embeddings)

print("\nFAISS Index 建立完成。")
print("Index 類型：", type(faiss_index).__name__)
print("Index 向量數量：", faiss_index.ntotal)
print("Index 向量維度：", faiss_index.d)

建立 FAISS Index
向量數量： 194
向量維度： 384

FAISS Index 建立完成。
Index 類型： IndexFlatIP
Index 向量數量： 194
Index 向量維度： 384


In [17]:
#將FAISS存成檔案
import json
import faiss
from pathlib import Path

BASE_DIR = Path("/content/scholarship_rag")
VECTOR_DB_DIR = BASE_DIR / "vector_DB"

FAISS_INDEX_FILE = VECTOR_DB_DIR / "faiss.index"
FAISS_INFO_FILE = VECTOR_DB_DIR / "faiss_info.json"

faiss.write_index(
    faiss_index,
    str(FAISS_INDEX_FILE)
)

faiss_info = {
    "index_type": "IndexFlatIP",
    "similarity_method": "Cosine Similarity",
    "total_vectors": int(faiss_index.ntotal),
    "vector_dimension": int(faiss_index.d),
    "normalized_vectors": True
}

with open(
    FAISS_INFO_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        faiss_info,
        file,
        ensure_ascii=False,
        indent=4
    )

print("FAISS Index 已儲存：", FAISS_INDEX_FILE)
print("FAISS 設定已儲存：", FAISS_INFO_FILE)

FAISS Index 已儲存： /content/scholarship_rag/vector_DB/faiss.index
FAISS 設定已儲存： /content/scholarship_rag/vector_DB/faiss_info.json


In [18]:
#建立Retrieval 函式
import json
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

BASE_DIR = Path("/content/scholarship_rag")
VECTOR_DB_DIR = BASE_DIR / "vector_DB"

CHUNK_FILE = VECTOR_DB_DIR / "chunks.json"
FAISS_INDEX_FILE = VECTOR_DB_DIR / "faiss.index"

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# 載入 Chunk
with open(CHUNK_FILE, "r", encoding="utf-8") as file:
    chunk_data = json.load(file)

if isinstance(chunk_data, dict) and "chunks" in chunk_data:
    chunks = chunk_data["chunks"]
else:
    chunks = chunk_data

# 載入 FAISS
faiss_index = faiss.read_index(
    str(FAISS_INDEX_FILE)
)

# 載入 Embedding Model
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print("=" * 70)
print("Retrieval 元件載入完成")
print("=" * 70)
print("Chunk 數量：", len(chunks))
print("FAISS 向量數量：", faiss_index.ntotal)
print("向量維度：", faiss_index.d)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retrieval 元件載入完成
Chunk 數量： 194
FAISS 向量數量： 194
向量維度： 384


In [19]:
#設定檢索用的函式
def retrieve_chunks(
    query,
    top_k=3
):
    """
    根據使用者問題，從 FAISS 找出最相似的 Chunk。
    """

    if not query or not query.strip():
        raise ValueError("查詢問題不能是空白。")

    if top_k <= 0:
        raise ValueError("top_k 必須大於 0。")

    # 將問題轉成 Embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # FAISS 相似度搜尋
    scores, indices = faiss_index.search(
        query_embedding,
        min(top_k, faiss_index.ntotal)
    )

    results = []

    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if index < 0 or index >= len(chunks):
            continue

        chunk = chunks[index]

        results.append(
            {
                "rank": rank,
                "score": float(score),
                "id": chunk.get("id", index),
                "source": chunk.get(
                    "source",
                    "未知來源"
                ),
                "page": chunk.get(
                    "page",
                    "未知頁碼"
                ),
                "chunk_number": chunk.get(
                    "chunk_number",
                    "未知"
                ),
                "text": chunk.get(
                    "text",
                    ""
                )
            }
        )

    return results


print("retrieve_chunks 函式建立完成。")

retrieve_chunks 函式建立完成。


In [20]:
#測試 Retrieval
question = "似鳥國際獎學金每名補助多少金額？"

results = retrieve_chunks(
    query=question,
    top_k=3
)

print("=" * 75)
print("查詢問題")
print("=" * 75)
print(question)

for result in results:
    print("\n" + "=" * 75)
    print(f"排名：{result['rank']}")
    print(f"相似度：{result['score']:.4f}")
    print(f"來源：{result['source']}")
    print(f"頁碼：第 {result['page']} 頁")
    print(f"Chunk：{result['chunk_number']}")
    print("-" * 75)
    print(result["text"][:700])

查詢問題
似鳥國際獎學金每名補助多少金額？

排名：1
相似度：0.6508
來源：似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁
Chunk：2
---------------------------------------------------------------------------
發放： 一、審查小組由學生事務長、生涯發展中心主任、課外活動組組長、生活輔導組組長及國 際事務處學術交流組組長等五人組成，由學生事務長擔任召集人。委員不克出席，可 由職務代理人代表出席。 二、獎學金分前後兩期(11 月及隔年 7 月)發放，每期發放新台幣 5 萬元。 三、獲獎學生應於隔年 7 月 20日前，繳交獲獎學年度之學業成績單及學習報告書，若未 如期繳交上述資料，學生應繳回已領之獎學金款項。 Article 7 Scholarship Selection and Disbursement: 1. The review committee consists of five members: the Dean of Student Affairs, the Director of the Career Development Center, the Chief of the Extracurricular Activities Section, the Chief of the Life Guidance Section, and the Chief of the Academic Exchange Section of the Office of International Affairs. The Dean of Student Affairs serves as the c

排名：2
相似度：0.6174
來源：林劉金珠女士勤學獎學金.pdf
頁碼：第 2 頁
Chunk：1
---------------------------------------------------------------------------
Article 4 The total amount of the Scholarship is NT$225,000 per semester, awarded to fifteen students, each receivi

In [21]:
#建立context函式
def build_context(
    retrieval_results,
    max_chars_per_chunk=1200
):
    """
    將檢索結果整理成提供給 LLM 的參考資料。
    """

    if not retrieval_results:
        return ""

    context_parts = []

    for result in retrieval_results:
        source = result.get(
            "source",
            "未知來源"
        )

        page = result.get(
            "page",
            "未知頁碼"
        )

        text = result.get(
            "text",
            ""
        ).strip()

        # 限制每個 Chunk 的文字長度
        text = text[:max_chars_per_chunk]

        context_part = (
            f"【參考資料 {result['rank']}】\n"
            f"文件名稱：{source}\n"
            f"頁碼：第 {page} 頁\n"
            f"相似度：{result['score']:.4f}\n"
            f"內容：\n{text}"
        )

        context_parts.append(context_part)

    return "\n\n".join(context_parts)


print("build_context 函式建立完成。")

build_context 函式建立完成。


In [41]:
#測試context
question = "似鳥國際獎學金每名補助多少金額？"

retrieval_results = retrieve_chunks(
    query=question,
    top_k=3
)

context = build_context(
    retrieval_results
)

print("=" * 75)
print("提供給 LLM 的 Context")
print("=" * 75)
print(context)

提供給 LLM 的 Context
【參考資料 1】
文件名稱：似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁
相似度：0.6508
內容：
發放： 一、審查小組由學生事務長、生涯發展中心主任、課外活動組組長、生活輔導組組長及國 際事務處學術交流組組長等五人組成，由學生事務長擔任召集人。委員不克出席，可 由職務代理人代表出席。 二、獎學金分前後兩期(11 月及隔年 7 月)發放，每期發放新台幣 5 萬元。 三、獲獎學生應於隔年 7 月 20日前，繳交獲獎學年度之學業成績單及學習報告書，若未 如期繳交上述資料，學生應繳回已領之獎學金款項。 Article 7 Scholarship Selection and Disbursement: 1. The review committee consists of five members: the Dean of Student Affairs, the Director of the Career Development Center, the Chief of the Extracurricular Activities Section, the Chief of the Life Guidance Section, and the Chief of the Academic Exchange Section of the Office of International Affairs. The Dean of Student Affairs serves as the c

【參考資料 2】
文件名稱：林劉金珠女士勤學獎學金.pdf
頁碼：第 2 頁
相似度：0.6174
內容：
Article 4 The total amount of the Scholarship is NT$225,000 per semester, awarded to fifteen students, each receiving NT$15,000. 第五條 申請條件： （一）各院系學生前學期學業總平均成績七十分以上及操行成績八十分以上者。 （二）家中收入因故銳減或家境清寒者（家庭年收入 70 萬以下）或政府登記之低 收入戶。 （三）未受公費待遇及未領其他獎學金者。 A rticle 5 Eligibility criter

In [42]:
#建立RAG的Prompt
def build_rag_prompt(question, context):
    prompt = f"""
你是一位「獎助學金規定問答助理」。

請嚴格遵守以下規則：

1. 只能根據下方提供的參考資料回答。
2. 不可以使用外部知識或自行猜測。
3. 若參考資料沒有明確答案，請回答：
   「根據目前提供的文件，無法確認此問題的答案。」
4. 優先使用與問題中的獎學金名稱相符的文件。
5. 不可混用不同獎學金的資格、金額、期限或申請方式。
6. 必須仔細區分「每期金額」與「每名總金額」。
7. 如果文件寫明分成數期發放，應計算所有期數的總額。
   例如：分兩期發放，每期5萬元，總額即為10萬元。
8. 計算金額時，請先確認期數與每期金額，再回答總額。
9. 回答使用繁體中文。
10. 答案最後列出文件名稱與頁碼。
11. 回答應簡潔完整，不需要列出無關文件。

以下是參考資料：

{context}

使用者問題：
{question}

請先分析文件中的期數、每期金額與總金額，再提供答案。
""".strip()

    return prompt


print("build_rag_prompt 函式建立完成。")

build_rag_prompt 函式建立完成。


In [43]:
#測試RAG Prompt
rag_prompt = build_rag_prompt(
    question=question,
    context=context
)

print("=" * 75)
print("RAG Prompt")
print("=" * 75)
print(rag_prompt)

RAG Prompt
你是一位「獎助學金規定問答助理」。

請嚴格遵守以下規則：

1. 只能根據下方提供的參考資料回答。
2. 不可以使用外部知識或自行猜測。
3. 若參考資料沒有明確答案，請回答：
   「根據目前提供的文件，無法確認此問題的答案。」
4. 優先使用與問題中的獎學金名稱相符的文件。
5. 不可混用不同獎學金的資格、金額、期限或申請方式。
6. 必須仔細區分「每期金額」與「每名總金額」。
7. 如果文件寫明分成數期發放，應計算所有期數的總額。
   例如：分兩期發放，每期5萬元，總額即為10萬元。
8. 計算金額時，請先確認期數與每期金額，再回答總額。
9. 回答使用繁體中文。
10. 答案最後列出文件名稱與頁碼。
11. 回答應簡潔完整，不需要列出無關文件。

以下是參考資料：

【參考資料 1】
文件名稱：似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁
相似度：0.6508
內容：
發放： 一、審查小組由學生事務長、生涯發展中心主任、課外活動組組長、生活輔導組組長及國 際事務處學術交流組組長等五人組成，由學生事務長擔任召集人。委員不克出席，可 由職務代理人代表出席。 二、獎學金分前後兩期(11 月及隔年 7 月)發放，每期發放新台幣 5 萬元。 三、獲獎學生應於隔年 7 月 20日前，繳交獲獎學年度之學業成績單及學習報告書，若未 如期繳交上述資料，學生應繳回已領之獎學金款項。 Article 7 Scholarship Selection and Disbursement: 1. The review committee consists of five members: the Dean of Student Affairs, the Director of the Career Development Center, the Chief of the Extracurricular Activities Section, the Chief of the Life Guidance Section, and the Chief of the Academic Exchange Section of the Office of International Affairs. The Dean of Student Affairs se

In [25]:
#GPU使用確認
import torch

print("=" * 70)
print("GPU 環境確認")
print("=" * 70)
print("CUDA 是否可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 名稱：", torch.cuda.get_device_name(0))
    print(
        "GPU 記憶體：",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("目前沒有使用 GPU，請在 Colab 切換成 GPU 執行階段。")

GPU 環境確認
CUDA 是否可用： True
GPU 名稱： Tesla T4
GPU 記憶體： 14.56 GB


In [26]:
#安裝Llama相關套件
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface-hub \
    sentencepiece

In [27]:
#登入Hugging Face
from huggingface_hub import notebook_login

notebook_login()

In [32]:
#設定模型名稱
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

print("準備載入模型：", MODEL_NAME)

準備載入模型： unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


In [33]:
#載入公開版 Llama 3.1
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

print("=" * 70)
print("載入公開版 Llama 3.1")
print("=" * 70)

print("正在載入 Tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("正在載入 Llama 3.1 8B Instruct 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.eval()

print("=" * 70)
print("Llama 模型載入完成")
print("=" * 70)
print("模型名稱：", MODEL_NAME)
print("主要裝置：", model.device)
print("資料型態：", model.dtype)

載入公開版 Llama 3.1
正在載入 Tokenizer...


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.70GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Llama 模型載入完成
模型名稱： unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
主要裝置： cuda:0
資料型態： torch.float16


In [44]:
#生成函式
def generate_llama_answer(
    prompt,
    max_new_tokens=250
):
    messages = [
        {
            "role": "system",
            "content": (
                "你是一位獎助學金規定問答助理。"
                "只能根據參考資料回答。"
                "必須區分單期金額與總金額，"
                "並正確整合文件中的期數與金額。"
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )

    model_inputs = {
        key: value.to(model.device)
        for key, value in model_inputs.items()
    }

    input_length = model_inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0, input_length:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    return answer

In [46]:
#測試回答
answer = generate_llama_answer(
    prompt=rag_prompt,
    max_new_tokens=250
)

print("=" * 75)
print("RAG 最終回答")
print("=" * 75)
print(answer)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG 最終回答
根據【參考資料 1】,似鳥國際獎學金每名每期發放新台幣 5 萬元，共分前後兩期發放。

因此，每名總金額為：2期 x 每期 5 萬元 = 10 萬元

答案：每名總金額為 10 萬元

文件名稱：似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁


In [47]:
#建立20個測試問題
test_questions = [
    {
        "id": 1,
        "question": "似鳥（NITORI）國際獎學金每名補助多少金額？"
    },
    {
        "id": 2,
        "question": "大立光電股份有限公司獎學金每位獲獎學生可獲得多少獎學金？"
    },
    {
        "id": 3,
        "question": "謝武志先生獎助學金一年共提供多少名額？"
    },
    {
        "id": 4,
        "question": "書卷獎每位得獎學生可獲得多少獎金？"
    },
    {
        "id": 5,
        "question": "校友總會獎助學金每名補助多少金額？"
    },
    {
        "id": 6,
        "question": "已經領取過正職員工薪資所得的學生，可以申請似鳥國際獎學金嗎？"
    },
    {
        "id": 7,
        "question": "延修生可以申請勤學獎助學金嗎？"
    },
    {
        "id": 8,
        "question": "已獲得政府各類獎助學金的學生，可以申請達達國際企業股份有限公司（Lagoon）弱勢獎學金嗎？"
    },
    {
        "id": 9,
        "question": "大學一年級學生可以申請建程科學科技工程數學獎學金嗎？"
    },
    {
        "id": 10,
        "question": "似鳥國際獎學金的申請資格有哪些？"
    },
    {
        "id": 11,
        "question": "達達國際企業股份有限公司（Lagoon）弱勢獎學金需要繳交哪些文件？"
    },
    {
        "id": 12,
        "question": "校友總會獎助學金的申請流程與應備文件有哪些？"
    },
    {
        "id": 13,
        "question": "謝武志先生獎助學金的申請資格與應備文件有哪些？"
    },
    {
        "id": 14,
        "question": "哪一種獎助學金最容易申請成功？"
    },
    {
        "id": 15,
        "question": "哪一項獎助學金的競爭最激烈？"
    },
    {
        "id": 16,
        "question": "如果同時符合多項獎助學金資格，可以全部一起申請嗎？"
    },
    {
        "id": 17,
        "question": "如果不同獎助學金文件對同一項規定有不同內容，應該以哪一份資料為準？"
    },
    {
        "id": 18,
        "question": "如果文件沒有提到外籍學生是否可以申請某項獎學金，系統應該如何回答？"
    },
    {
        "id": 19,
        "question": "似鳥國際獎學金每名補助多少金額？請區分每期金額與每名總金額。"
    },
    {
        "id": 20,
        "question": "達達國際企業股份有限公司（Lagoon）弱勢獎學金第二學期是否一定會繼續發放？"
    }
]

print("測試題數量：", len(test_questions))

for item in test_questions:
    print(f"第 {item['id']:02d} 題：{item['question']}")

測試題數量： 20
第 01 題：似鳥（NITORI）國際獎學金每名補助多少金額？
第 02 題：大立光電股份有限公司獎學金每位獲獎學生可獲得多少獎學金？
第 03 題：謝武志先生獎助學金一年共提供多少名額？
第 04 題：書卷獎每位得獎學生可獲得多少獎金？
第 05 題：校友總會獎助學金每名補助多少金額？
第 06 題：已經領取過正職員工薪資所得的學生，可以申請似鳥國際獎學金嗎？
第 07 題：延修生可以申請勤學獎助學金嗎？
第 08 題：已獲得政府各類獎助學金的學生，可以申請達達國際企業股份有限公司（Lagoon）弱勢獎學金嗎？
第 09 題：大學一年級學生可以申請建程科學科技工程數學獎學金嗎？
第 10 題：似鳥國際獎學金的申請資格有哪些？
第 11 題：達達國際企業股份有限公司（Lagoon）弱勢獎學金需要繳交哪些文件？
第 12 題：校友總會獎助學金的申請流程與應備文件有哪些？
第 13 題：謝武志先生獎助學金的申請資格與應備文件有哪些？
第 14 題：哪一種獎助學金最容易申請成功？
第 15 題：哪一項獎助學金的競爭最激烈？
第 16 題：如果同時符合多項獎助學金資格，可以全部一起申請嗎？
第 17 題：如果不同獎助學金文件對同一項規定有不同內容，應該以哪一份資料為準？
第 18 題：如果文件沒有提到外籍學生是否可以申請某項獎學金，系統應該如何回答？
第 19 題：似鳥國際獎學金每名補助多少金額？請區分每期金額與每名總金額。
第 20 題：達達國際企業股份有限公司（Lagoon）弱勢獎學金第二學期是否一定會繼續發放？


In [48]:
#測試建立單一題的問答
import time


def answer_one_question(
    question,
    top_k=3,
    max_new_tokens=300
):
    """
    執行單一問題的完整 RAG 流程。

    參數：
    question：使用者問題
    top_k：FAISS 取回的文件片段數量
    max_new_tokens：模型最多生成多少個新 Token

    回傳：
    question：原始問題
    answer：模型回答
    retrieval_results：FAISS 找到的文件根據
    elapsed_seconds：執行時間
    """

    start_time = time.time()

    # 1. 使用 FAISS 檢索相關文件
    retrieval_results = retrieve_chunks(
        query=question,
        top_k=top_k
    )

    # 2. 將檢索結果整理成 Context
    context = build_context(
        retrieval_results
    )

    # 3. 建立 RAG Prompt
    rag_prompt = build_rag_prompt(
        question=question,
        context=context
    )

    # 4. 使用 Llama 3.1 生成答案
    answer = generate_llama_answer(
        prompt=rag_prompt,
        max_new_tokens=max_new_tokens
    )

    # 5. 計算本題執行時間
    elapsed_seconds = round(
        time.time() - start_time,
        2
    )

    # 6. 回傳結果
    return {
        "question": question,
        "answer": answer,
        "retrieval_results": retrieval_results,
        "elapsed_seconds": elapsed_seconds
    }


print("answer_one_question 函式建立完成。")

answer_one_question 函式建立完成。


In [49]:
#測試第一題問答
single_result = answer_one_question(
    question=test_questions[0]["question"],
    top_k=3,
    max_new_tokens=250
)

print("=" * 80)
print("測試問題")
print("=" * 80)
print(single_result["question"])

print("\n" + "=" * 80)
print("RAG 回答")
print("=" * 80)
print(single_result["answer"])

print("\n" + "=" * 80)
print("回答根據")
print("=" * 80)

for rank, result in enumerate(
    single_result["retrieval_results"],
    start=1
):
    print(f"\n【參考資料 {rank}】")
    print("文件名稱：", result["source"])
    print("頁碼：第", result["page"], "頁")
    print("相似度：", round(float(result["score"]), 4))
    print("內容：")
    print(result["text"])
    print("-" * 80)

print(
    "\n執行時間：",
    single_result["elapsed_seconds"],
    "秒"
)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


測試問題
似鳥（NITORI）國際獎學金每名補助多少金額？

RAG 回答
根據【參考資料 1】,似鳥（NITORI）國際獎學金每名每期發放新台幣 5 萬元，共分前後兩期發放。

因此，每名總金額為：2期 x 新台幣 5 萬元/期 = 新台幣 10 萬元

文件名稱：似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁

回答根據

【參考資料 1】
文件名稱： 似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁
相似度： 0.754
內容：
發放： 一、審查小組由學生事務長、生涯發展中心主任、課外活動組組長、生活輔導組組長及國 際事務處學術交流組組長等五人組成，由學生事務長擔任召集人。委員不克出席，可 由職務代理人代表出席。 二、獎學金分前後兩期(11 月及隔年 7 月)發放，每期發放新台幣 5 萬元。 三、獲獎學生應於隔年 7 月 20日前，繳交獲獎學年度之學業成績單及學習報告書，若未 如期繳交上述資料，學生應繳回已領之獎學金款項。 Article 7 Scholarship Selection and Disbursement: 1. The review committee consists of five members: the Dean of Student Affairs, the Director of the Career Development Center, the Chief of the Extracurricular Activities Section, the Chief of the Life Guidance Section, and the Chief of the Academic Exchange Section of the Office of International Affairs. The Dean of Student Affairs serves as the c
--------------------------------------------------------------------------------

【參考資料 2】
文件名稱： 大立光電股份有限公司獎學金.pdf
頁碼：第 1 頁
相似度： 0.6733
內容：
neering, totaling 10 students.

In [50]:
#正式問答20題
import gc
import torch

all_test_results = []

print("=" * 90)
print("開始執行 20 題 RAG 測試")
print("=" * 90)

for item in test_questions:

    question_id = item["id"]
    question = item["question"]

    print("\n" + "=" * 90)
    print(f"正在執行第 {question_id:02d} 題")
    print("=" * 90)
    print("問題：", question)

    try:
        result = answer_one_question(
            question=question,
            top_k=3,
            max_new_tokens=250
        )

        result["id"] = question_id
        result["status"] = "成功"

        all_test_results.append(result)

        print("\n【RAG 回答】")
        print(result["answer"])

        print("\n【回答根據】")

        for rank, evidence in enumerate(
            result["retrieval_results"],
            start=1
        ):
            print(f"\n【參考資料 {rank}】")
            print("文件名稱：", evidence["source"])
            print("頁碼：第", evidence["page"], "頁")
            print(
                "相似度：",
                round(float(evidence["score"]), 4)
            )
            print("內容：")
            print(evidence["text"])
            print("-" * 90)

        print(
            "\n本題執行時間：",
            result["elapsed_seconds"],
            "秒"
        )

    except Exception as error:

        print("\n本題執行失敗：", str(error))

        all_test_results.append({
            "id": question_id,
            "question": question,
            "answer": "",
            "retrieval_results": [],
            "elapsed_seconds": 0,
            "status": f"失敗：{str(error)}"
        })

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 90)
print("20 題測試執行完成")
print("=" * 90)

success_count = sum(
    result["status"] == "成功"
    for result in all_test_results
)

failure_count = len(all_test_results) - success_count

print("成功題數：", success_count)
print("失敗題數：", failure_count)

開始執行 20 題 RAG 測試

正在執行第 01 題
問題： 似鳥（NITORI）國際獎學金每名補助多少金額？


[transformers] Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



【RAG 回答】
根據【參考資料 1】,似鳥（NITORI）國際獎學金每名每期發放新台幣 5 萬元，共分前後兩期發放。

因此，每名總金額為：2期 x 新台幣 5 萬元/期 = 新台幣 10 萬元

文件名稱：似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁

【回答根據】

【參考資料 1】
文件名稱： 似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁
相似度： 0.754
內容：
發放： 一、審查小組由學生事務長、生涯發展中心主任、課外活動組組長、生活輔導組組長及國 際事務處學術交流組組長等五人組成，由學生事務長擔任召集人。委員不克出席，可 由職務代理人代表出席。 二、獎學金分前後兩期(11 月及隔年 7 月)發放，每期發放新台幣 5 萬元。 三、獲獎學生應於隔年 7 月 20日前，繳交獲獎學年度之學業成績單及學習報告書，若未 如期繳交上述資料，學生應繳回已領之獎學金款項。 Article 7 Scholarship Selection and Disbursement: 1. The review committee consists of five members: the Dean of Student Affairs, the Director of the Career Development Center, the Chief of the Extracurricular Activities Section, the Chief of the Life Guidance Section, and the Chief of the Academic Exchange Section of the Office of International Affairs. The Dean of Student Affairs serves as the c
------------------------------------------------------------------------------------------

【參考資料 2】
文件名稱： 大立光電股份有限公司獎學金.pdf
頁碼：第 1 頁
相似度： 0.6733
內容：
neering, totaling 10 students. 三、獎助金額：獲獎學生每學年度

[transformers] Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



正在執行第 02 題
問題： 大立光電股份有限公司獎學金每位獲獎學生可獲得多少獎學金？


...


【RAG 回答】
根據目前提供的文件，無法確認此問題的答案。

因為沒有任何文件提供相關的規定或說明。

如果您能夠提供更多的文件或資訊，我們將會更好地幫助您。

【回答根據】

【參考資料 1】
文件名稱： 書卷獎.pdf
頁碼：第 2 頁
相似度： 0.6238
內容：
 accordingly.
------------------------------------------------------------------------------------------

【參考資料 2】
文件名稱： 南加州校友會獎學金.pdf
頁碼：第 3 頁
相似度： 0.583
內容：
same procedure for amendments.
------------------------------------------------------------------------------------------

【參考資料 3】
文件名稱： 張慧高女士紀念獎助學金.pdf
頁碼：第 3 頁
相似度： 0.5405
內容：
ame procedure applies for amendments.
------------------------------------------------------------------------------------------

本題執行時間： 9.37 秒

正在執行第 18 題
問題： 如果文件沒有提到外籍學生是否可以申請某項獎學金，系統應該如何回答？


[transformers] Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



【RAG 回答】
根據提供的文件，我們可以分析如下：

1. 文件【參考資料 3】中，第五條提到「本獎助學金每學年度每人最高給與金額為 30 萬元」，並且「本獎助學金每學年度每人最高給與金額為 30 萬元，分為 10 個月發放，每月 3 萬元」。
2. 根據這個信息，我們可以計算每期金額和總金額：
 * 每期金額：3 萬元
 * 總金額：3 萬元 x 10 個月 = 30 萬元

因此，答案是：

根據目前提供的文件，勤學獎助學金每學年度每人最高給與金額為 30 萬元，分為 10 個月發放，每月 3 萬元。

文件名稱：勤學獎助學金.pdf
頁碼：第 1 頁

【回答根據】

【參考資料 1】
文件名稱： 似鳥(NITORI)國際獎學金.pdf
頁碼：第 4 頁
相似度： 0.6656
內容：
4. The student cannot accept the scholarship due to objective reasons or cannot continue their studies. 第九條 無正當理由不參加公益財團法人似鳥國際獎學財團主辦之交流活動時，喪失獲獎資格，並 需返還領取之獎學金。 Article 9 Failure to participate in exchange activities organized by the Nitori International Scholarship Foundation without a valid reason will result in the loss of scholarship eligibility and require the return of received scholarship funds 第十條 學生獲獎紀錄永久保存，申請資料則保存 1 年。 Article 10 The records of awarded students are permanently retained, while application materials are kept for one year. 第十一條 本辦法經公益財團法人似鳥國際獎學財團同意，陳請校長核定後實施，修正時亦同。 Articl
----------------------------------------------

[transformers] Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



【RAG 回答】
根據【參考資料 1】,似鳥國際獎學金的發放情況如下：

* 每期發放新台幣 5 萬元。
* 總共有兩期發放。

因此，每名學生的每期金額為新台幣 5 萬元，每名總金額為新台幣 10 萬元（兩期發放）。

文件名稱：似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁

【回答根據】

【參考資料 1】
文件名稱： 似鳥(NITORI)國際獎學金.pdf
頁碼：第 3 頁
相似度： 0.6414
內容：
發放： 一、審查小組由學生事務長、生涯發展中心主任、課外活動組組長、生活輔導組組長及國 際事務處學術交流組組長等五人組成，由學生事務長擔任召集人。委員不克出席，可 由職務代理人代表出席。 二、獎學金分前後兩期(11 月及隔年 7 月)發放，每期發放新台幣 5 萬元。 三、獲獎學生應於隔年 7 月 20日前，繳交獲獎學年度之學業成績單及學習報告書，若未 如期繳交上述資料，學生應繳回已領之獎學金款項。 Article 7 Scholarship Selection and Disbursement: 1. The review committee consists of five members: the Dean of Student Affairs, the Director of the Career Development Center, the Chief of the Extracurricular Activities Section, the Chief of the Life Guidance Section, and the Chief of the Academic Exchange Section of the Office of International Affairs. The Dean of Student Affairs serves as the c
------------------------------------------------------------------------------------------

【參考資料 2】
文件名稱： 林劉金珠女士勤學獎學金.pdf
頁碼：第 2 頁
相似度： 0.5818
內容：
Article 4 The total amount of the Sc

In [51]:
#將20題問答匯出成檔案
import pandas as pd


table_rows = []

for result in all_test_results:

    retrieval_results = result.get(
        "retrieval_results",
        []
    )

    # 取 Top-1 作為主要回答根據
    if len(retrieval_results) > 0:

        main_result = retrieval_results[0]

        main_source = main_result["source"]
        main_page = main_result["page"]
        main_score = round(
            float(main_result["score"]),
            4
        )
        main_text = main_result["text"]

    else:

        main_source = ""
        main_page = ""
        main_score = ""
        main_text = ""

    table_rows.append({
        "題號": result["id"],
        "測試問題": result["question"],
        "RAG回答": result["answer"],
        "主要參考文件": main_source,
        "主要參考頁碼": main_page,
        "主要相似度": main_score,
        "主要文件內容": main_text,
        "執行時間（秒）": result["elapsed_seconds"],
        "執行狀態": result["status"]
    })


results_df = pd.DataFrame(table_rows)

pd.set_option(
    "display.max_colwidth",
    300
)

display(results_df)

,題號,測試問題,RAG回答,主要參考文件,主要參考頁碼,主要相似度,主要文件內容,執行時間（秒）,執行狀態
0,1,似鳥（NITORI）國際獎學金每名補助多少金額？,"根據【參考資料 1】,似鳥（NITORI）國際獎學金每名每期發放新台幣 5 萬元，共分前後兩期發放。\n\n因此，每名總金額為：2期 x 新台幣 5 萬元/期 = 新台幣 10 萬元\n\n文件名稱：似鳥(NITORI)國際獎學金.pdf\n頁碼：第 3 頁",似鳥(NITORI)國際獎學金.pdf,3,0.7540,發放： 一、審查小組由學生事務長、生涯發展中心主任、課外活動組組長、生活輔導組組長及國 際事務處學術交流組組長等五人組成，由學生事務長擔任召集人。委員不克出席，可 由職務代理人代表出席。 二、獎學金分前後兩期(11 月及隔年 7 月)發放，每期發放新台幣 5 萬元。 三、獲獎學生應於隔年 7 月 20日前，繳交獲獎學年度之學業成績單及學習報告書，若未 如期繳交上述資料，學生應繳回已領之獎學金款項。 Article 7 Scholarship Selection and Disbursement: 1. The review committee consists of five memb...,16.55,成功
1,2,大立光電股份有限公司獎學金每位獲獎學生可獲得多少獎學金？,根據【參考資料 3】大立光電股份有限公司獎學金.pdf第 1 頁，該獎學金每學年度頒給 10 名學生，每名學生可獲得新台幣 1 萬元整。\n\n因此，大立光電股份有限公司獎學金每位獲獎學生可獲得新台幣 1 萬元整。\n\n文件名稱：大立光電股份有限公司獎學金.pdf\n頁碼：第 1 頁,似鳥(NITORI)國際獎學金.pdf,1,0.7071,"larship. 第三條 獎學金名額及金額：每年甄選5名學生，每名獎學金新台幣 10 萬元，得備取 1 至 2 名。 Article 3 Number and Amount of Scholarships: Five students are selected each year, each receiving NT$100,000. One to two alternates may be selected. 第四條 申請資格：本校在學之本國籍大學部 3 年級、4 年級、碩士班 1 年級、2 年級學生(在職 生、延畢生、曾經領取過正職員工薪資所得者及預定未來 1 年內出國交換或研修 ...",19.80,成功
2,3,謝武志先生獎助學金一年共提供多少名額？,"根據【參考資料 1】,謝武志先生獎助學金一年共提供12名額。\n\n第五條中提到： ""以上每年合計頒發名額：12 名，共頒發獎學金新臺幣30 萬元整。""\n\n因此，謝武志先生獎助學金一年共提供12名額。\n\n文件名稱：武志先生獎助學金.pdf\n頁碼：第 1 頁",武志先生獎助學金.pdf,1,0.6862,1 國立中興大學謝武志先生獎助學金辦法 114 年 9 月 4 日訂定 第一條 本校EMBA 領袖組謝武志學長為鼓勵經濟弱勢學子，激發其向上精神， 順利完成學業，特設置「謝武志先生獎助學金」 。 第二條 獎助金額及名額： (一)碩士班獎助2 名學生，每名新臺幣5 萬元整。 (二)大學部獎助10 名學生，每名新臺幣2 萬元整。 以上每年合計頒發名額：12 名，共頒發獎學金新臺幣30 萬元整。 第三條 本獎助學金之保管存放，由國立中興大學學務處會同主計室及總務處等 相關單位辦理。 第四條 本校本國籍學生(不含在職專班)符合下列情形之一者： （一）低收入戶。 （二）中低收入戶。 （三）特殊境...,20.34,成功
3,4,書卷獎每位得獎學生可獲得多少獎金？,"根據參考資料 1 和參考資料 2，書卷獎每學期頒發獎狀乙紙及獎金新台幣三千元整。\n\n因此，書卷獎每位得獎學生的獎金為每學期新台幣三千元整。\n\n根據參考資料 1，書卷獎每學期辦理一次，因此每學期頒發獎金共計 3,000 x 1 = 3,000 元。\n\n根據參考資料 2，書卷獎每學期得獎學生頒發獎狀乙紙及獎金新台幣三千元整。\n\n因此，書卷獎每位得獎學生的獎金為每學期新台幣三千元整。\n\n文件名稱：書卷獎.pdf\n頁碼：第 2 頁\n\n注意：根據參考資料 1 和參考資料 2 的描述，書卷獎每學期頒發獎金新台幣三千元整，因此無需計算總金額。",書卷獎.pdf,2,0.7474,"、 本獎項每學期得獎學生頒發獎狀乙紙及獎金新台幣三千元整。 IV. A winning student for t he semester shall be granted a certificate and prize money of NT$3,000. 五、 本獎項每學期辦理一次 ， 由教務處註冊組提供各學系前一學期學業成績名冊與班級 人數，學生事務處生活輔導組辦理審核作業。 V. The Award is organized every semester. The Registration Division, Office of Academic Affairs, shall ...",35.46,成功
4,5,校友總會獎助學金每名補助多少金額？,根據目前提供的文件，無法確認「校友總會獎助學金」的資格、金額、期限或申請方式。\n\n文件中並未提及「校友總會獎助學金」的相關資訊，因此無法提供確切的答案。\n\n建議查詢相關文件或聯繫提供者以取得更多資訊。,林劉金珠女士勤學獎學金.pdf,2,0.7318,"Article 4 The total amount of the Scholarship is NT$225,000 per semester, awarded to fifteen students, each receiving NT$15,000. 第五條 申請條件： （一）各院系學生前學期學業總平均成績七十分以上及操行成績八十分以上者。 （二）家中收入因故銳減或家境清寒者（家庭年收入 70 萬以下）或政府登記之低 收入戶。 （三）未受公費待遇及未領其他獎學金者。 A rticle 5 Eligibility criteria: 1. Students from each co...",14.50,成功
5,6,已經領取過正職員工薪資所得的學生，可以申請似鳥國際獎學金嗎？,"根據目前提供的文件，無法確認此問題的答案。\n\n因為在【參考資料 3】中，文件並未提及正職員工薪資所得的學生是否可以申請似鳥國際獎學金。\n\n但是在【參考資料 3】,第 3 頁有提到，獎學金分前後兩期發放，每期發放新台幣 5 萬元。\n\n因此，假設獎學金共有 2 期，則每名總金額為 2 x 50,000 = 100,000 新台幣。\n\n文件名稱：似鳥(NITORI)國際獎學金.pdf\n頁碼：第 3 頁",勤學獎助學金.pdf,2,0.5708,"nts apply with the average grades from their previous school, conduct grades not included. 5. Students who have received other scholarships totaling NT$20,000 or more are not eligible to apply. 6. Students who receive more than 20,000 yuan in educational subsidies per academic year are not eligi...",23.48,成功
6,7,延修生可以申請勤學獎助學金嗎？,根據參考資料 2 的文件，延修生或因修習雙主修、輔系而延長就學學生不得申請勤學獎助學金。\n\n因此，答案是：延修生不能申請勤學獎助學金。\n\n文件名稱：勤學獎助學金.pdf\n頁碼：第 1 頁,助學功德金.pdf,2,0.6710,arship Review Committee based on donation amounts and budget availability. 第六條 本助 學金之申請者應本著自助人助的方式，請先申請學雜費減免或就學貸款後，若 生活需求尚有不足之部分，始得申請本助學金。獲得本助學金者，應於畢業後就業 時，視個人能力回饋原受贊助金額，以使本助學金得以永續，造福更多的清寒學子。 Article 6 This aid will only provide to students who still cannot afford daily expenses after getting a ...,16.14,成功
7,8,已獲得政府各類獎助學金的學生，可以申請達達國際企業股份有限公司（Lagoon）弱勢獎學金嗎？,根據目前提供的文件，無法確認此問題的答案。\n\n因為文件中沒有提到達達國際企業股份有限公司（Lagoon）弱勢獎學金的資格、金額、期限或申請方式。\n\n因此，無法確定已獲得政府各類獎助學金的學生是否可以申請達達國際企業股份有限公司（Lagoon）弱勢獎學金。\n\n文件名稱：陳守先生紀念獎助學金.pdf\n頁碼：第 1 頁\n\n文件名稱：武志先生獎助學金.pdf\n頁碼：第 2 頁\n\n文件名稱：勤學獎助學金.pdf\n頁碼：第 1 頁,陳守先生紀念獎助學金.pdf,1,0.7137,tions are eligib

In [52]:
#匯出成CSV
csv_path = "/content/RAG_20題測試結果.csv"

results_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("CSV 已儲存：", csv_path)

CSV 已儲存： /content/RAG_20題測試結果.csv


In [53]:
#下載CSV
from google.colab import files

files.download(
    "/content/RAG_20題測試結果.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>